# Grade 12 · Unit 5 — Fairness audit

> **SYNTHETIC** health table. The generator gives mountain mothers lower facility use; a real audit would ask why (distance? roads? staff?).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

h = pd.read_csv(data_file("health-synthetic", "health_synthetic.csv"))
FEATURES = ["antenatal_visits", "wealth_quintile", "mother_age"]
X, y = h[FEATURES], h["institutional_delivery"]
print(h.shape, "| share delivering in a facility:", round(float(y.mean()), 3))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score
idx_tr, idx_te = train_test_split(h.index, test_size=0.4, random_state=2, stratify=y)
m = LogisticRegression().fit(X.loc[idx_tr], y.loc[idx_tr])
test = h.loc[idx_te].copy()
test["pred"] = m.predict(X.loc[idx_te])
print(test["ecological_belt"].value_counts().to_dict())

## Compare by group
**Predict:** is the smallest group served worst?

In [ ]:
rows = []
for belt, g in test.groupby("ecological_belt"):
    rows.append(dict(belt=belt, n=len(g), actual_rate=round(g["institutional_delivery"].mean(), 3),
                     selection_rate=round(g["pred"].mean(), 3),
                     accuracy=round((g["pred"] == g["institutional_delivery"]).mean(), 3),
                     recall=round(recall_score(g["institutional_delivery"], g["pred"]), 3)))
audit = pd.DataFrame(rows)
assert len(audit) == 3
audit

## Reading the audit
* Are accuracy or recall noticeably lower for one group? Is the group small (so numbers are shaky)?
* Does the model predict a different *share of yes* for each group — is that fair or does it copy past inequality?
* What decision would the model feed, and who could be harmed by a miss or a false alarm?

**Make:** write a half-page audit: findings, limits (synthetic, small groups), and three recommendations (e.g. collect more data from the under-served group, set group-aware thresholds only with community agreement, keep a human in the loop).